# SL ladder · 00 · Inputs, and what they can and cannot support

An independent Sri Lanka-only rebuild of the M0-M5 ladder, from the raw inputs on this machine.

The frozen study built its exposure from **ERA5-Land at 0.1 deg**, pulled from CDS with a key.
That is not on this Mac. What *is* here is a cached **ERA5 0.25 deg** window and **CHIRPS 0.05 deg**,
staged for WP5. So this rebuild substitutes a coarser temperature/humidity product and keeps
CHIRPS for rainfall.

That substitution is the single largest deviation in the series, and it is deliberate: it lets the
whole path be exercised end-to-end today, and `sl_03` measures exactly what it costs against the
3,926 frozen predictions. Everything else is reconstructed to match.

This notebook only *inventories and checks* inputs. It writes nothing.

## 1 · Setup

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd

REPO = Path.cwd().parent if Path.cwd().name.startswith("notebooks") else Path.cwd()
DQ   = REPO / "data_quarantine"
ERA  = DQ / "wp5_exposure" / "era5_0p25_window"
CHI  = DQ / "wp5_exposure" / "chirps_window"
OUTD = DQ / "sl_ladder"
OUTD.mkdir(parents=True, exist_ok=True)

WER  = DQ / "wer_srilanka/frozen/wer_dengue_currentweek_rdhs_2018_2025_v2.1-refresh.csv"
POP  = DQ / "m6_geomatics/m6_batchB_population_by_year.csv"
WERA = DQ / "wp5_exposure/wp5_buildB_weights_era5_0p25_srilanka_2020.csv"
WCHI = DQ / "wp5_exposure/wp5_buildB_weights_chirps_p05_srilanka_2020.csv"
FROZEN = REPO / "ALT_STATS/frozen/srilanka_matched_pairs.csv"

for p in [WER, POP, WERA, WCHI, FROZEN]:
    assert p.exists(), f"missing input: {p}"
    print(f"{p.stat().st_size/1e6:8.2f} MB  {p.relative_to(REPO)}")

## 2 · The outcome series

Weekly notified dengue by RDHS district from the Weekly Epidemiological Report. Two extractions are
on disk (`frozen/…v2.1-refresh` and `processed_quarantine/…v2`); they are byte-identical on the case
column, so the choice does not matter. 59 district-weeks failed text-layer extraction and are
flagged, not imputed.

In [ ]:
o = pd.read_csv(WER).rename(columns={"year": "epi_year", "week": "epi_week"})
print(o.shape, "| districts", o.rdhs.nunique(), "| years", o.epi_year.min(), "-", o.epi_year.max())
print(o.extraction_flag.value_counts().to_string())

alt = pd.read_csv(DQ / "wer_srilanka/processed_quarantine/wer_dengue_rdhs_2018_2025_quarantine_v2.csv")
k = ["year", "week", "rdhs"]
same = (o.rename(columns={"epi_year": "year", "epi_week": "week"}).set_index(k).sort_index()["dengue_current_week"]
        .fillna(-1).equals(alt.set_index(k).sort_index()["dengue_current_week"].fillna(-1)))
print("\nthe two extractions agree on every case count:", same)

## 3 · The climate grids

ERA5 is hourly `t2m`/`d2m` on an 18x11 window; CHIRPS is daily precipitation on 79x45. The `window`
field carries the grid offsets, which `sl_01` needs to line the arrays up with the district weights.
Checked here rather than assumed, because a silent off-by-one in the grid index would produce
plausible-looking climate for the wrong place.

In [ ]:
z = np.load(ERA / "era5_0p25_t2m_2018.npz")
r0, r1, c0, c1 = z["window"]
print("ERA5 arrays:", {k: z[k].shape for k in z})
print(f"window rows {r0}-{r1}, cols {c0}-{c1}")
print(f"implied extent: lat {90-r1*0.25}..{90-r0*0.25} N, lon {c0*0.25}..{c1*0.25} E")
assert 5 < 90 - r1 * 0.25 < 8 and 78 < c0 * 0.25 < 82, "window is not over Sri Lanka"

c = np.load(CHI / "chirps_p05_srilanka_2018.npz")
print("\nCHIRPS arrays:", {k: c[k].shape for k in c})
print("years on disk: ERA5", sorted(p.stem[-4:] for p in ERA.glob("*t2m*.npz")))
print("               CHIRPS", sorted(p.stem[-4:] for p in CHI.glob("*.npz")))

## 4 · District weights

WP5 already computed, for each RDHS district, the set of grid cells covering it and their `w_area`
(area share) and `w_pop` (population share). That file is what makes this rebuild possible without
the RDHS boundary polygons, which are **not** on this machine.

`w_area` reproduces the original area-mean build; `w_pop` is WP5's Build B. This series uses
`w_area`, so the ladder is comparable to the frozen run rather than to a WP5 variant.

In [ ]:
we, wc = pd.read_csv(WERA), pd.read_csv(WCHI)
print("ERA5 weights  ", we.shape, "| districts", we.geometry_id.nunique())
print("CHIRPS weights", wc.shape, "| districts", wc.geometry_id.nunique())
for nm, w in [("era5", we), ("chirps", wc)]:
    s = w.groupby("geometry_id").w_area.sum()
    print(f"  {nm}: w_area sums to 1 per district: {np.allclose(s, 1)}")

xw = we[["geometry_id", "rdhs_name"]].drop_duplicates()
print("\nname crosswalk covers every WER district:", set(o.rdhs) == set(xw.rdhs_name))

## 5 · Population denominators

WorldPop R2025A is local for **2018-2020 only**. `sl_02` shows that the frozen run behaved as though
the denominator were time-invariant, so the missing years turn out not to matter — but that is a
finding, not an assumption made here.

In [ ]:
pop = pd.read_csv(POP)
print("years:", sorted(pop.year.unique()), "| districts", pop.rdhs_name.nunique())
print(pop.groupby("year").pop_sum.sum().div(1e6).round(2).to_string(), "  (millions)")

## 6 · The fidelity target

`ALT_STATS/frozen/srilanka_matched_pairs.csv` holds the frozen run's 3,926 test-set predictions for
the full and no-climate models. It is the only local artefact that can tell us whether this rebuild
lands in the right place, and `sl_03` is built entirely around it.

In [ ]:
fz = pd.read_csv(FROZEN, parse_dates=["predictor_week", "target_week"])
fz = fz[fz.setting == "SriLanka"]
print(fz.shape, "| districts", fz.spatial_unit_id.nunique())
print("predictor weeks:", fz.predictor_week.min().date(), "->", fz.predictor_week.max().date())
print("all pairs are predictor + 28d:", (fz.target_week - fz.predictor_week).dt.days.eq(28).all())
print("predictor weekday:", fz.predictor_week.dt.day_name().unique().tolist())
print("prevalence:", round(fz.outcome.mean(), 6))

## 7 · What this series can and cannot claim

| Rung | Status |
|---|---|
| M1, M4, M5, M5_no-climate | **Reconstruction.** Design and fit ported verbatim from `analysis/v12_referee_response/run/sl_matched_and_recal.py`. |
| M0, M2, M3 | **Reimplementation.** No in-repo Sri Lanka code exists; built from the Methods spec. Weaker provenance. |
| Exposure | **Substituted.** ERA5 0.25 deg stands in for ERA5-Land 0.1 deg. Cost measured in `sl_03`. |

An exact bit-match is not attainable on this machine and is not the goal. The goal is a rebuild whose
every departure from the frozen run is identified and quantified.